In [ ]:
# Install required packages
!pip install -q transformers datasets evaluate scikit-learn gradio accelerate

In [ ]:
import numpy as np
import pandas as pd
import torch

from datasets import load_dataset
from transformers import (
    BertTokenizerFast,
    BertForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import accuracy_score, f1_score
import gradio as gr

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
import sys
sys.modules["torchvision"] = None  # prevents datasets from trying to use it

In [ ]:
!pip install -q -U datasets transformers accelerate

In [ ]:
dataset = load_dataset("fancyzhx/ag_news")
print(dataset)
print(dataset["train"][0])

In [ ]:
# Optional: smaller subset so training finishes quickly on free Colab GPU
train_dataset = dataset["train"].shuffle(seed=42).select(range(8000))
test_dataset  = dataset["test"].shuffle(seed=42).select(range(2000))

In [ ]:
tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")

def tokenize_function(example):
    # max_length=64 is excellent for keeping things fast and light!
    return tokenizer(example["text"], padding="max_length", truncation=True, max_length=64)

# --- REMOVED THE OVERWRITING LINES HERE ---

# Now this correctly maps over your 8,000 and 2,000 row subsets
train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset  = test_dataset.map(tokenize_function, batched=True)

# Set format for PyTorch
train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])

In [ ]:
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=4
).to(device)

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average="weighted")
    return {"accuracy": acc, "f1": f1}

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,  # <-- Highly recommended to add this!
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none"
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()


In [ ]:
results = trainer.evaluate()
print(results)

In [ ]:
model.save_pretrained("./news_classifier_model")
tokenizer.save_pretrained("./news_classifier_model")

In [ ]:
labels_map = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}

def predict_news_category(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=64).to(device)
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
    pred = torch.argmax(outputs.logits, dim=-1).item()
    return labels_map[pred]

# Quick test
predict_news_category("Apple unveils new AI chip for its next-gen iPhones")

In [ ]:
demo = gr.Interface(
    fn=predict_news_category,
    inputs=gr.Textbox(lines=2, placeholder="Enter a news headline..."),
    outputs="text",
    title="News Topic Classifier (BERT)",
    description="Classifies news headlines into World, Sports, Business, or Sci/Tech using a fine-tuned BERT model."
)

demo.launch()